# 🧪 話者分離の実験（pyannote Community-1）

**最終更新日時: 2026-10-09 12:02**

本体（`notebook.ipynb`）は **変更しません**。このノートブックは実験専用です。
要件: `docs/spec_話者分離実験.md` ／ 計画: `docs/roadmap_話者分離実験.md`

## 使い方
1. **ランタイムを GPU（T4）に変更する**（「ランタイム」→「ランタイムのタイプを変更」）
2. Colab の「シークレット」に `HF_TOKEN`（Hugging Face の Read トークン）を登録して、ノートブックからのアクセスをオンにする
3. 上から順にセルを実行する（Step 1 のインストールのあと、ランタイムが自動で再起動します）

## 守っていること
- `01_input` の音声は **コピーするだけ**（移動・削除しません）
- Gemini・Notion・スプレッドシートは使いません

## 進み具合
- [x] Step 1: 土台（インストール・認証・音声の用意）
- [x] Step 2: 条件 A（人数のヒントなし）
- [x] Step 3: 条件 B（人数を指定）と比較
- [x] Step 4: 文字起こしと突き合わせ（Colab で確認済み。話者の順番は合うが、文字の時間が崩れて発言がこま切れになる）
- [x] Step 5: 話者で先に区切ってから文字起こし
- [ ] 長時間（2.5〜4時間）の音声での確認: `docs/roadmap_長時間音声確認.md`（Step 1-2 の設定 `SOURCE_FILE_NAME` / `EXPERIMENT_SUBDIR` / `MAX_MINUTES` を使う。Step 3・4 は飛ばす）


## Step 1-1: インストール
`pyannote.audio`（話者分離）と `whisperx`（文字起こし）を入れます（初回は数分かかります）。入れ終わると、ランタイムが自動で再起動されます。


In [ ]:
# ライブラリのインストール
# ※ 初回実行時は数分かかります
import time
start = time.time()

# uv（高速パッケージマネージャ）を使って一括インストール
!pip install -q uv
!uv pip install "pyannote.audio>=4.0" whisperx huggingface_hub "numba>=0.61"

elapsed = int(time.time() - start)
print(f'✅ ライブラリのインストール完了（{elapsed}秒）')

# 🔔 完了音で通知（ド・ミ・ソの上昇音）
from IPython.display import Javascript, display
display(Javascript("""
var ctx = new AudioContext();
[440, 550, 660].forEach((freq, i) => {
  var osc = ctx.createOscillator();
  var gain = ctx.createGain();
  osc.connect(gain); gain.connect(ctx.destination);
  osc.frequency.value = freq;
  gain.gain.setValueAtTime(0.3, ctx.currentTime + i * 0.15);
  gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + i * 0.15 + 0.3);
  osc.start(ctx.currentTime + i * 0.15);
  osc.stop(ctx.currentTime + i * 0.15 + 0.3);
});
"""))

print('🔄 ランタイムを自動で再起動します（数秒後に再接続してください）...')

# インストール後に自動でランタイムを再起動（手動再起動の代わり）
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


---
## Step 1-2: 🔑 設定と認証
Google Drive のマウントと、Hugging Face のトークン・Community-1 の利用条件の確認をします。
ポップアップが出たら「許可」を押してください。


In [ ]:
import os
import shutil
import subprocess

from google.colab import drive, userdata

# ========================================
# 1. 設定
# ========================================
BASE_DIR        = '/content/drive/MyDrive/MeetingTranscript'
INPUT_AUDIO_DIR = f'{BASE_DIR}/01_input'
EXPERIMENT_ROOT = f'{BASE_DIR}/04_temp/experiment_diarization'  # 実験の結果の保存先の大元（Colabが切れても残る）
WORK_DIR        = '/content/work'  # 音声のコピー先（Colab側の一時領域）
DIARIZATION_MODEL = 'pyannote/speaker-diarization-community-1'

# 実験で使う人数（条件 B で指定する。今回の音声は男性2人）
NUM_SPEAKERS = 2

# --- 長時間の音声で試すための設定 ---
# 使う音声のファイル名。None なら 01_input のファイル名順で先頭の1本
SOURCE_FILE_NAME = None
# 結果の保存先の下に作るフォルダ名。音声ごとに変える（同じにすると、別の音声の保存を読んでしまう）
# 空文字 '' なら、今まで通り EXPERIMENT_ROOT 直下（22分の音声の結果）
EXPERIMENT_SUBDIR = 'long'
# 音声の先頭から何分だけ使うか。None なら全部。まず 60 で試し、動いたら None にして全体を試す
MAX_MINUTES = 60

EXPERIMENT_DIR = f'{EXPERIMENT_ROOT}/{EXPERIMENT_SUBDIR}' if EXPERIMENT_SUBDIR else EXPERIMENT_ROOT

# ========================================
# 2. HF_TOKEN の読み込み
# ========================================
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as e:
    raise ValueError(
        '❌ Colab のシークレット HF_TOKEN が読み込めません。\n'
        '   左のカギアイコン → HF_TOKEN を登録し、「ノートブックからのアクセス」をオンにしてください。\n'
        f'   （詳細: {e}）'
    )
if not HF_TOKEN:
    raise ValueError('❌ HF_TOKEN が空です。Hugging Face の Read トークンを登録してください。')
print('✅ HF_TOKEN 読み込み完了')

# ========================================
# 3. GPU と ffmpeg の確認
# ========================================
import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU が使えません。「ランタイム」→「ランタイムのタイプを変更」で T4 GPU にしてください（CPU でも動きますが遅いです）')

missing_tools = [tool for tool in ('ffmpeg', 'ffprobe') if shutil.which(tool) is None]
if missing_tools:
    raise RuntimeError(
        f'❌ {missing_tools} が見つかりません。\n'
        f'   新しいセルで !apt-get install -y ffmpeg を実行してから、もう一度このセルを実行してください。'
    )
print('✅ ffmpeg / ffprobe 確認OK')

# ========================================
# 4. Google Drive マウント
# ========================================
drive.mount('/content/drive')
print('✅ Google Drive マウント完了')

# ========================================
# 5. Community-1 の利用条件に同意済みか確認
#    （モデルの設定ファイルを1つ取得してみる。同意していないと GatedRepoError になる）
# ========================================
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import EntryNotFoundError, GatedRepoError, HfHubHTTPError

try:
    hf_hub_download(DIARIZATION_MODEL, 'config.yaml', token=HF_TOKEN)
    print(f'✅ {DIARIZATION_MODEL} にアクセスできます')
except EntryNotFoundError:
    # 設定ファイルの名前が違うだけで、モデル自体にはアクセスできている
    print(f'✅ {DIARIZATION_MODEL} にアクセスできます（設定ファイル名の確認はスキップ）')
except GatedRepoError:
    raise RuntimeError(
        f'❌ {DIARIZATION_MODEL} の利用条件に同意していません。\n'
        f'   https://huggingface.co/{DIARIZATION_MODEL} を開いて同意してから、もう一度実行してください。'
    )
except HfHubHTTPError as e:
    raise RuntimeError(
        f'❌ Hugging Face へのアクセスに失敗しました（トークンが無効・期限切れの可能性があります）。\n'
        f'   詳細: {e}'
    )


---
## Step 1-3: 音声の用意
`01_input` の音声を **コピー**して、話者分離に使う形式（16kHz・モノラルの wav）に変換します。
元のファイルは動かしません。音声の長さも表示します。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['INPUT_AUDIO_DIR', 'WORK_DIR', 'EXPERIMENT_DIR', 'SOURCE_FILE_NAME', 'MAX_MINUTES', 'shutil'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

# 対応する拡張子（本体と同じ考え方。動画も音声として読める）
AUDIO_EXTENSIONS = ('.mp3', '.m4a', '.wav', '.flac', '.ogg', '.aac', '.mp4', '.mov', '.avi', '.mkv', '.webm')

# 1. 入力フォルダから音声を探す（SOURCE_FILE_NAME の指定があればそれ、なければファイル名順で先頭の1つ）
if not os.path.isdir(INPUT_AUDIO_DIR):
    raise FileNotFoundError(f'❌ 入力フォルダが見つかりません: {INPUT_AUDIO_DIR}')

audio_files = sorted(
    name for name in os.listdir(INPUT_AUDIO_DIR)
    if name.lower().endswith(AUDIO_EXTENSIONS)
)
if not audio_files:
    raise FileNotFoundError(f'❌ {INPUT_AUDIO_DIR} に音声ファイルがありません')

if SOURCE_FILE_NAME is None:
    source_name = audio_files[0]
    print(f'🎧 使う音声: {source_name}（{len(audio_files)}件中の先頭）')
else:
    if SOURCE_FILE_NAME not in audio_files:
        raise FileNotFoundError(
            f'❌ 指定した音声が見つかりません: {SOURCE_FILE_NAME}\n'
            f'   {INPUT_AUDIO_DIR} にあるファイル: {audio_files}'
        )
    source_name = SOURCE_FILE_NAME
    print(f'🎧 使う音声: {source_name}（指定）')

# 2. Colab 側にコピー（Drive の元ファイルは動かさない）
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
source_ext = os.path.splitext(source_name)[1].lower()
copied_path = f'{WORK_DIR}/source{source_ext}'  # ファイル名の空白や日本語を避けるため固定名にする
shutil.copy2(f'{INPUT_AUDIO_DIR}/{source_name}', copied_path)
print('✅ コピー完了')

# 3. 16kHz・モノラルの wav に変換（-vn: 動画の映像は捨てる、-t: 先頭から指定した秒数だけ使う）
AUDIO_PATH = f'{WORK_DIR}/audio.wav'
ffmpeg_command = ['ffmpeg', '-y', '-i', copied_path, '-vn', '-ac', '1', '-ar', '16000']
if MAX_MINUTES:
    ffmpeg_command += ['-t', str(int(MAX_MINUTES * 60))]
    print(f'✂️ 先頭の {MAX_MINUTES} 分だけ使います（MAX_MINUTES）')
result = subprocess.run(ffmpeg_command + [AUDIO_PATH], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f'❌ ffmpeg の変換に失敗しました:\n{result.stderr[-1500:]}')

# 4. 長さを調べて表示
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', AUDIO_PATH],
    capture_output=True, text=True,
)
try:
    AUDIO_DURATION_SEC = float(probe.stdout.strip())
except ValueError:
    raise RuntimeError(f'❌ 音声の長さが取得できませんでした:\n{probe.stderr[-500:]}')

minutes, seconds = divmod(int(AUDIO_DURATION_SEC), 60)
print(f'✅ 変換完了: {AUDIO_PATH}')
print(f'⏱️ 音声の長さ: {minutes}分{seconds}秒（{AUDIO_DURATION_SEC:.1f}秒）')
print(f'📁 結果の保存先: {EXPERIMENT_DIR}')
print(f'💾 wav のサイズ: {os.path.getsize(AUDIO_PATH) / 1024**3:.2f} GB（読み込むとメモリにこの約2倍乗る）')


---
## Step 2-1: Community-1 の読み込みと、使う関数の準備
モデルを読み込みます（初回はダウンロードがあります）。
このセルで作る関数は、条件 B（Step 3）でもそのまま使います。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['DIARIZATION_MODEL', 'HF_TOKEN', 'AUDIO_PATH', 'AUDIO_DURATION_SEC', 'EXPERIMENT_DIR'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

import json
import time
import wave

import numpy as np
import torch
from pyannote.audio import Pipeline

# ========================================
# 1. モデルの読み込み
# ========================================
print('⏳ Community-1 を読み込み中...')
pipeline = Pipeline.from_pretrained(DIARIZATION_MODEL, token=HF_TOKEN)
if pipeline is None:
    raise RuntimeError('❌ モデルを読み込めませんでした。トークンと利用条件の同意を確認してください。')
if torch.cuda.is_available():
    pipeline.to(torch.device('cuda'))
print('✅ モデルの読み込み完了')


# ========================================
# 2. 使う関数
# ========================================
def load_waveform(path):
    # 16bit の wav を読み込み、pyannote に渡せる形（チャンネル数×サンプル数）にする
    # （ファイル名のまま渡すと音声ライブラリの相性で失敗することがあるので、自分で読み込む）
    with wave.open(path, 'rb') as wf:
        sample_rate = wf.getframerate()
        n_channels = wf.getnchannels()
        sample_width = wf.getsampwidth()
        frames = wf.readframes(wf.getnframes())
    if sample_width != 2:
        raise ValueError(f'❌ 16bit の wav を想定していますが、{sample_width * 8}bit でした')
    data = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
    data = data.reshape(-1, n_channels).T
    return torch.from_numpy(np.ascontiguousarray(data)), sample_rate


def annotation_to_segments(annotation):
    # pyannote の結果を、保存しやすい「開始・終了・話者」の一覧に直す
    return [
        {'start': round(turn.start, 3), 'end': round(turn.end, 3), 'speaker': speaker}
        for turn, _, speaker in annotation.itertracks(yield_label=True)
    ]


def run_diarization(num_speakers=None):
    # 話者分離を1回実行する。num_speakers を渡すと、その人数で分けようとする
    waveform, sample_rate = load_waveform(AUDIO_PATH)
    options = {} if num_speakers is None else {'num_speakers': num_speakers}

    start = time.time()
    output = pipeline({'waveform': waveform, 'sample_rate': sample_rate}, **options)
    elapsed = time.time() - start

    # pyannote 4.x は結果をまとめたオブジェクトで返す（古い形式なら Annotation がそのまま返る）
    annotation = getattr(output, 'speaker_diarization', output)
    result = {
        'num_speakers_option': num_speakers,
        'elapsed_sec': round(elapsed, 1),
        'audio_duration_sec': round(AUDIO_DURATION_SEC, 1),
        'segments': annotation_to_segments(annotation),
    }
    # 同時に話している部分を除いた版（文字起こしとの突き合わせで使う）
    exclusive = getattr(output, 'exclusive_speaker_diarization', None)
    if exclusive is not None:
        result['exclusive_segments'] = annotation_to_segments(exclusive)
    return result


def format_time(seconds):
    hours, rest = divmod(int(seconds), 3600)
    minutes, secs = divmod(rest, 60)
    return f'{hours}:{minutes:02d}:{secs:02d}'


def show_summary(result, max_rows=40):
    # 検出人数・話者ごとの発話時間・話者が入れ替わった回数・区間の先頭を表示する
    segments = sorted(result['segments'], key=lambda s: s['start'])
    if not segments:
        print('⚠️ 区間が1つも検出されませんでした')
        return

    totals = {}
    for seg in segments:
        totals[seg['speaker']] = totals.get(seg['speaker'], 0.0) + (seg['end'] - seg['start'])
    # 話者が変わった回数は、同時発話を除いた版で数える（重なった区間を別々に数えると多く見えるため）
    exclusive = sorted(result.get('exclusive_segments') or segments, key=lambda s: s['start'])
    changes = sum(1 for a, b in zip(exclusive, exclusive[1:]) if a['speaker'] != b['speaker'])

    label = '指定なし' if result['num_speakers_option'] is None else f"{result['num_speakers_option']}人を指定"
    print(f'■ 条件: {label}')
    print(f"⏱️ 処理時間: {result['elapsed_sec']}秒（音声 {result['audio_duration_sec']}秒）")
    print(f'👥 検出された人数: {len(totals)}人 ／ 区間の数: {len(segments)}（同時発話を除くと {len(exclusive)}） ／ 話者が変わった回数（同時発話を除く）: {changes}')
    total_sec = sum(totals.values())
    for speaker, sec in sorted(totals.items()):
        print(f'   {speaker}: {format_time(sec)}（{sec:.0f}秒、{sec / total_sec * 100:.0f}%）')

    print(f'--- 先頭 {min(max_rows, len(segments))} 区間 ---')
    for seg in segments[:max_rows]:
        print(f"{format_time(seg['start'])} - {format_time(seg['end'])}  {seg['speaker']}")


def save_result(result, file_name):
    # 一時ファイルに書いてから名前を変える（書き込み中に切れても、壊れたファイルが残らない）
    path = f'{EXPERIMENT_DIR}/{file_name}'
    tmp_path = path + '.tmp'
    with open(tmp_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=1, default=float)
    os.replace(tmp_path, path)
    print(f'💾 保存しました: {path}')


def compare_results(result_x, result_y, step=0.1, top_n=10):
    # 2つの結果を、0.1秒ごとの「誰が話しているか」に直して比べる
    # （SPEAKER_00 / 01 の番号は結果ごとに入れ替わることがあるので、一致が多くなる対応を自動で選ぶ）
    from itertools import permutations

    total_sec = max(result_x['audio_duration_sec'], result_y['audio_duration_sec'])
    n_frames = int(total_sec / step) + 2

    def to_frames(result):
        segs = result.get('exclusive_segments') or result['segments']
        names = sorted({s['speaker'] for s in segs})
        frames = np.full(n_frames, -1, dtype=int)  # -1 = 誰も話していない
        for seg in segs:
            frames[int(seg['start'] / step): int(seg['end'] / step) + 1] = names.index(seg['speaker'])
        return frames, names

    frames_x, names_x = to_frames(result_x)
    frames_y, names_y = to_frames(result_y)
    print(f'人数: {len(names_x)}人 と {len(names_y)}人')
    if len(names_x) > len(names_y):
        print('⚠️ 人数が違うので、話者ごとの対応づけは省略します')
        return

    both = (frames_x >= 0) & (frames_y >= 0)
    best_mapping, best_agree = None, -1
    for perm in permutations(range(len(names_y)), len(names_x)):
        lookup = np.array(list(perm) + [-1])  # 最後の要素は「誰も話していない（-1）」用
        agree = int(((lookup[frames_x] == frames_y) & both).sum())
        if agree > best_agree:
            best_mapping, best_agree = lookup, agree

    mapped_x = best_mapping[frames_x]
    both_sec = both.sum() * step
    disagree = both & (mapped_x != frames_y)
    print(f'両方が話者を検出した時間: {both_sec:.0f}秒')
    print(f'   一致: {best_agree * step:.0f}秒（{best_agree / max(both.sum(), 1) * 100:.1f}%）')
    print(f'   食い違い: {disagree.sum() * step:.0f}秒')
    print(f'片方だけが話者ありとした時間: {((frames_x >= 0) != (frames_y >= 0)).sum() * step:.0f}秒')

    # 食い違いが長く続いた場所（聞き比べるときの目印）
    edges = np.flatnonzero(np.diff(np.concatenate(([0], disagree.astype(int), [0]))))
    spans = sorted(zip(edges[::2], edges[1::2]), key=lambda se: se[1] - se[0], reverse=True)[:top_n]
    if spans:
        print(f'--- 食い違いが長い場所 上位{len(spans)}（聞き比べの目印） ---')
        for s_idx, e_idx in sorted(spans):
            print(f'{format_time(s_idx * step)} - {format_time(e_idx * step)}（{(e_idx - s_idx) * step:.1f}秒）')


def show_memory(label):
    # Colab の RAM と GPU メモリの使用量を表示する（長時間の音声で、収まるかを確かめるため）
    try:
        import psutil
        vm = psutil.virtual_memory()
        print(f'🧮 [{label}] RAM: {(vm.total - vm.available) / 1024**3:.1f} / {vm.total / 1024**3:.1f} GB 使用')
    except ImportError:
        print(f'🧮 [{label}] RAM: psutil が無いので表示できません')
    if torch.cuda.is_available():
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'🧮 [{label}] GPU: 今 {torch.cuda.memory_allocated() / 1024**3:.1f} GB ／ 最大 {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB（全体 {total:.1f} GB）')


def exclusive_of(result):
    return result.get('exclusive_segments') or result['segments']


def show_dialogue(turns, max_rows=60):
    # 発言の数・短い発言の数・話者ごとの文字数を表示して、先頭の会話を見せる
    short_turns = sum(1 for t in turns if len(t['text'].strip()) <= 5)
    print(f'💬 発言の数: {len(turns)} ／ 5文字以下の短い発言: {short_turns}')
    chars = {}
    for t in turns:
        chars[t['speaker']] = chars.get(t['speaker'], 0) + len(t['text'])
    for speaker, count in sorted(chars.items()):
        print(f'   {speaker}: {count}文字')
    print(f'--- 先頭 {min(max_rows, len(turns))} 発言 ---')
    for t in turns[:max_rows]:
        print(f"[{format_time(t['start'])}] {t['speaker']}：{t['text'].strip()}")


def save_dialogue(turns, file_name):
    # 一時ファイルに書いてから名前を変える
    path = f'{EXPERIMENT_DIR}/{file_name}'
    tmp_path = path + '.tmp'
    with open(tmp_path, 'w', encoding='utf-8') as f:
        for t in turns:
            f.write(f"[{format_time(t['start'])}] {t['speaker']}：{t['text'].strip()}\n")
    os.replace(tmp_path, path)
    print(f'💾 保存しました: {path}')


print('✅ 関数の準備完了')


---
## Step 2-2: 条件 A（人数のヒントなし）
前回（v1 系）と同じ条件で、話者分離を実行します。22分の音声なら約1分、4時間の音声なら10分前後の見込みです（処理時間とメモリを表示します）。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['pipeline', 'run_diarization'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

show_memory('話者分離の前')
print('⏳ 条件 A: 話者分離を実行中（人数は指定しません）...')
result_a = run_diarization(num_speakers=None)
show_memory('話者分離のあと')

show_summary(result_a)
save_result(result_a, 'diarization_A.json')


---
## Step 3: 条件 B（人数を指定）と比較
`NUM_SPEAKERS`（今回は2人）を指定して、もう一度実行します。そのあと、条件 A と並べて比べます。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['result_a', 'run_diarization', 'NUM_SPEAKERS'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

print(f'⏳ 条件 B: 話者分離を実行中（人数は {NUM_SPEAKERS} 人を指定）...')
result_b = run_diarization(num_speakers=NUM_SPEAKERS)

show_summary(result_b, max_rows=20)
save_result(result_b, 'diarization_B.json')

print()
print('=' * 40)
print('条件 A と条件 B の比較')
print('=' * 40)
print(f"A: {len({s['speaker'] for s in result_a['segments']})}人検出 ／ B: {len({s['speaker'] for s in result_b['segments']})}人検出")
compare_results(result_a, result_b)


---
## Step 4-1: 文字起こし（WhisperX）
本体と同じ設定（`large-v3`・日本語）で文字起こしして、**文字ごとの時間**を付けます（アライメント）。
22分の音声で数分かかります。結果は Drive に保存するので、2回目からは保存済みの結果を読み込みます。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['AUDIO_PATH', 'AUDIO_DURATION_SEC', 'save_result', 'format_time'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

import gc

import whisperx

WHISPER_FILE = 'whisper_aligned.json'
WHISPER_PATH = f'{EXPERIMENT_DIR}/{WHISPER_FILE}'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'

if os.path.exists(WHISPER_PATH):
    # 保存済みなら、文字起こしはやり直さない
    with open(WHISPER_PATH, encoding='utf-8') as f:
        whisper_segments = json.load(f)
    print(f'♻️ 保存済みの文字起こしを読み込みました（{len(whisper_segments)}区間）')
else:
    print('🎙️ WhisperX で文字起こし中...')
    start = time.time()
    audio = whisperx.load_audio(AUDIO_PATH)

    model = whisperx.load_model('large-v3', device, compute_type=compute_type)
    transcribed = model.transcribe(audio, batch_size=4, language='ja')
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ 文字起こし完了（{int(time.time() - start)}秒、{len(transcribed['segments'])}区間）")

    # 文字ごとの時間を付ける（話者との突き合わせに使う）
    print('📏 文字ごとの時間を付けています...')
    align_model, align_metadata = whisperx.load_align_model(language_code='ja', device=device)
    aligned = whisperx.align(transcribed['segments'], align_model, align_metadata, audio, device)
    del align_model
    gc.collect()
    torch.cuda.empty_cache()

    whisper_segments = aligned['segments']
    save_result(whisper_segments, WHISPER_FILE)

# 文字数のチェック（日本語の会話は毎分300〜400文字くらい）
total_chars = sum(len(seg.get('text', '')) for seg in whisper_segments)
chars_per_minute = total_chars / (AUDIO_DURATION_SEC / 60)
print(f'📝 文字数: {total_chars}（毎分 {chars_per_minute:.0f} 文字）')
if chars_per_minute < 20:
    print('⚠️ 文字数が少なすぎます。文字起こしに失敗している可能性があります')
print('--- 先頭3区間 ---')
for seg in whisper_segments[:3]:
    print(f"{format_time(seg['start'])} {seg['text']}")


---
## Step 4-2: 話者と文字起こしを突き合わせる
文字ごとに、**時間が一番重なる話者**を割り当てて、同じ話者が続く部分をまとめます。
同時発話を除いた版（`exclusive_segments`）を使います。結果は `result_A.txt` に保存します。
条件 B が A とまったく同じなら、B は省略します。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['whisper_segments', 'result_a', 'result_b'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

from bisect import bisect_right


def segment_items(whisper_segments):
    # 文字起こしを「文字ごと（時間つき）」の一覧にする。文字ごとの時間がない区間は、区間ごと1つにする
    items = []
    for seg in whisper_segments:
        words = seg.get('words') or []
        if words:
            for w in words:
                items.append({'text': w.get('word', ''), 'start': w.get('start'), 'end': w.get('end')})
        else:
            items.append({'text': seg.get('text', ''), 'start': seg.get('start'), 'end': seg.get('end')})
    return items


def find_speaker(start, end, speaker_segments, starts):
    # 時間が一番重なる話者を返す。どの区間とも重ならないときは、一番近い区間の話者にする
    best_speaker, best_overlap = None, 0.0
    right = bisect_right(starts, end)
    for j in range(right - 1, -1, -1):
        seg = speaker_segments[j]
        if seg['end'] <= start:
            break  # これより左の区間は、さらに前に終わっているので重ならない
        overlap = min(end, seg['end']) - max(start, seg['start'])
        if overlap > best_overlap:
            best_speaker, best_overlap = seg['speaker'], overlap
    if best_speaker is not None:
        return best_speaker

    neighbors = []
    if right > 0:
        neighbors.append(speaker_segments[right - 1])
    if right < len(speaker_segments):
        neighbors.append(speaker_segments[right])
    if not neighbors:
        return None
    nearest = min(neighbors, key=lambda seg: max(seg['start'] - end, start - seg['end'], 0.0))
    return nearest['speaker']


def build_dialogue(whisper_segments, speaker_segments):
    # 文字起こしと話者を突き合わせて、同じ話者が続く部分を1つの発言にまとめる
    speaker_segments = sorted(speaker_segments, key=lambda s: s['start'])
    starts = [s['start'] for s in speaker_segments]

    turns = []
    last_speaker, last_end = None, 0.0
    for item in segment_items(whisper_segments):
        if item['start'] is None or item['end'] is None:
            # 時間が取れなかった文字（数字・記号など）は、直前の話者にくっつける
            speaker, start, end = last_speaker, last_end, last_end
        else:
            speaker = find_speaker(item['start'], item['end'], speaker_segments, starts)
            start, end = item['start'], item['end']
        if speaker is None:
            speaker = last_speaker or 'UNKNOWN'

        if turns and turns[-1]['speaker'] == speaker:
            turns[-1]['text'] += item['text']
            turns[-1]['end'] = max(turns[-1]['end'], end)
        else:
            turns.append({'speaker': speaker, 'start': start, 'end': end, 'text': item['text']})
        last_speaker, last_end = speaker, end
    return turns


turns_a = build_dialogue(whisper_segments, exclusive_of(result_a))
print('■ 条件 A')
show_dialogue(turns_a)
save_dialogue(turns_a, 'result_A.txt')

if exclusive_of(result_b) == exclusive_of(result_a):
    print('\nℹ️ 条件 B は条件 A とまったく同じ結果なので、result_B.txt は作りません')
else:
    turns_b = build_dialogue(whisper_segments, exclusive_of(result_b))
    print('\n■ 条件 B')
    show_dialogue(turns_b, max_rows=20)
    save_dialogue(turns_b, 'result_B.txt')


---
## Step 5: 話者で先に区切ってから文字起こし
Step 4 は「全体を文字起こし → 文字に時間を付ける → 話者に当てはめる」でしたが、話者が速く入れ替わる所で、文字の時間が崩れました。
ここでは順番を逆にします。**話者分離の結果で先に音声を切って、切った音声ごとに文字起こし**します。文字に時間を付ける処理は使いません。

- 5-1: クリップ（文字起こしする単位）に切り分ける。設定値はこのセルの先頭にあります
- 5-2: クリップごとに文字起こし（22分の音声で数分。保存しながら進むので、途切れても続きから再開できます）
- 5-3: 会話の形にして `result_A2.txt` に保存。Step 4 の `result_A.txt` と比べます

※ Step 3（条件 B）と Step 4（全体の文字起こし）は飛ばして、Step 2-2 のあとにそのまま Step 5 へ進めます。長時間の音声では、飛ばしてください。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['result_a', 'AUDIO_DURATION_SEC', 'format_time'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

# ========================================
# 切り方の設定（結果を見て調整する）
# ========================================
MERGE_GAP_SEC = 2.0  # 同じ話者が続くとき、この秒数以内の間ならつないで1つのクリップにする（1.0 だと、冒頭の短い声が拾えなかった）
PAD_SEC = 0.15       # 切り出すとき、前後に足す余白（秒）。大きいと隣の人の声が混ざりやすい（0.3 だと言葉が重なった。0.15 は実験中）
MIN_CLIP_SEC = 0.5   # これより短い声は文字起こししない（相づちなど。短すぎると幻覚が出やすいため）


def build_clips(speaker_segments, merge_gap, pad, min_clip, total_sec):
    # 話者ごとの区間を、文字起こしする単位（クリップ）にする
    segments = sorted(speaker_segments, key=lambda s: (s['start'], s['end']))

    # 同じ話者が続いていて、間が短いものをつなぐ（間に別の話者の声があるときは、つながない）
    merged = []
    for seg in segments:
        last = merged[-1] if merged else None
        if last and last['speaker'] == seg['speaker'] and seg['start'] - last['end'] <= merge_gap:
            last['end'] = max(last['end'], seg['end'])
        else:
            merged.append({'speaker': seg['speaker'], 'start': seg['start'], 'end': seg['end']})

    clips = []
    for item in merged:
        clips.append({
            'speaker': item['speaker'],
            'start': item['start'],
            'end': item['end'],
            'cut_start': max(0.0, item['start'] - pad),       # 切り出し開始（余白つき）
            'cut_end': min(total_sec, item['end'] + pad),     # 切り出し終わり（余白つき）
            'transcribe': item['end'] - item['start'] >= min_clip,  # 短すぎる声は文字起こししない
        })
    return clips


exclusive_segments_a = result_a.get('exclusive_segments') or result_a['segments']
clips = build_clips(exclusive_segments_a, MERGE_GAP_SEC, PAD_SEC, MIN_CLIP_SEC, AUDIO_DURATION_SEC)

to_transcribe = [c for c in clips if c['transcribe']]
short_clips = [c for c in clips if not c['transcribe']]
print(f'✂️ 設定: つなぐ間隔 {MERGE_GAP_SEC}秒 ／ 余白 {PAD_SEC}秒 ／ 短い声の基準 {MIN_CLIP_SEC}秒')
print(f'元の区間: {len(exclusive_segments_a)}件 → クリップ: {len(clips)}件')
print(f'   文字起こしする: {len(to_transcribe)}件（合計 {sum(c["end"] - c["start"] for c in to_transcribe) / 60:.1f}分）')
print(f'   短い声（文字起こししない）: {len(short_clips)}件')
print(f'   一番長いクリップ: {max(c["end"] - c["start"] for c in clips):.1f}秒')


### Step 5-2: クリップごとに文字起こし


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['clips', 'to_transcribe', 'AUDIO_PATH', 'format_time', 'show_memory'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

import gc

import numpy as np
import whisperx

CLIP_TEXT_FILE = 'clip_texts.json'
CLIP_TEXT_PATH = f'{EXPERIMENT_DIR}/{CLIP_TEXT_FILE}'
SAMPLE_RATE = 16000  # 音声の用意で 16kHz に変換してある
RETRY_EMPTY_MIN_SEC = 2.0  # この秒数以上のクリップなのに文字が空のときだけ、条件を変えてもう一度試す（短い声は、本当に空のことがある）
RETRY_WIDE_PAD_SEC = 1.5   # 2回目の再挑戦で、前後に足す広めの余白（秒）
RETRIED_SUFFIX = '#retried'  # 再挑戦ずみの印（保存ファイルのキーの末尾に付ける。何度も再挑戦しないため）
device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'


def clip_key(clip):
    # 切り出す範囲が同じなら、保存済みの文字起こしを使い回す（切り方を変えたクリップだけやり直す）
    return f"{clip['cut_start']:.2f}-{clip['cut_end']:.2f}"


def transcribe_chunk(model, chunk):
    result = model.transcribe(chunk, batch_size=4, language='ja')
    return ''.join(seg['text'] for seg in result['segments']).strip()


def normalize_volume(chunk):
    # 音が小さいクリップを、最大音量が 0.95 になるまで大きくする（最大 20 倍まで）
    peak = float(np.abs(chunk).max()) if len(chunk) else 0.0
    if peak <= 0:
        return chunk
    return (chunk * min(0.95 / peak, 20.0)).astype(np.float32)


def retry_empty_clip(model, audio, clip):
    # 文字が空だったクリップを、①音量を上げて ②さらに前後を広げて、順に再挑戦する。戻り値: (文字, 出た方法)
    cut = audio[int(clip['cut_start'] * SAMPLE_RATE): int(clip['cut_end'] * SAMPLE_RATE)]
    text = transcribe_chunk(model, normalize_volume(cut))
    if text:
        return text, '音量を上げて'
    wide_start = max(0.0, clip['start'] - RETRY_WIDE_PAD_SEC)
    wide_end = min(AUDIO_DURATION_SEC, clip['end'] + RETRY_WIDE_PAD_SEC)
    wide = audio[int(wide_start * SAMPLE_RATE): int(wide_end * SAMPLE_RATE)]
    text = transcribe_chunk(model, normalize_volume(wide))
    if text:
        return text, '前後を広げて'
    return '', None


def is_long(clip):
    return clip['end'] - clip['start'] >= RETRY_EMPTY_MIN_SEC


def save_clip_texts(texts):
    # 一時ファイルに書いてから名前を変える（途中で切れても、壊れたファイルが残らない）
    tmp_path = CLIP_TEXT_PATH + '.tmp'
    with open(tmp_path, 'w', encoding='utf-8') as f:
        json.dump(texts, f, ensure_ascii=False, indent=1)
    os.replace(tmp_path, CLIP_TEXT_PATH)


clip_texts = {}
if os.path.exists(CLIP_TEXT_PATH):
    with open(CLIP_TEXT_PATH, encoding='utf-8') as f:
        clip_texts = json.load(f)

todo = [c for c in clips if c['transcribe'] and clip_key(c) not in clip_texts]
retry_only = [
    c for c in clips
    if c['transcribe'] and clip_key(c) in clip_texts and not clip_texts[clip_key(c)]
    and is_long(c) and clip_key(c) + RETRIED_SUFFIX not in clip_texts
]
print(f'📝 文字起こしするクリップ: {len(todo)}件（保存済みを再利用: {len(to_transcribe) - len(todo)}件）')
if retry_only:
    print(f'🔁 保存済みで文字が空だった長いクリップ（{RETRY_EMPTY_MIN_SEC}秒以上）を再挑戦: {len(retry_only)}件')

if todo or retry_only:
    print('🎙️ WhisperX でクリップごとに文字起こし中...')
    start_time = time.time()
    audio = whisperx.load_audio(AUDIO_PATH)
    model = whisperx.load_model('large-v3', device, compute_type=compute_type)
    show_memory('文字起こしの開始時')

    errors = 0
    rescued, still_empty = [], []
    jobs = [(c, False) for c in todo] + [(c, True) for c in retry_only]  # (クリップ, 再挑戦だけか)
    for i, (clip, only_retry) in enumerate(jobs, 1):
        try:
            if only_retry:
                text = ''
            else:
                chunk = audio[int(clip['cut_start'] * SAMPLE_RATE): int(clip['cut_end'] * SAMPLE_RATE)]
                text = transcribe_chunk(model, chunk)
                clip_texts[clip_key(clip)] = text

            if not text and is_long(clip):
                # 長いのに空なのは不自然。音量を上げる・前後を広げる、で再挑戦する
                text, how = retry_empty_clip(model, audio, clip)
                clip_texts[clip_key(clip)] = text
                clip_texts[clip_key(clip) + RETRIED_SUFFIX] = '1'
                if text:
                    rescued.append((clip['start'], how, text))
                else:
                    still_empty.append(clip['start'])
        except Exception as e:
            # 失敗したクリップは保存しない（次に実行したとき、もう一度試す）
            errors += 1
            print(f"⚠️ {format_time(clip['start'])} のクリップで失敗: {e}")

        if i % 25 == 0 or i == len(jobs):
            save_clip_texts(clip_texts)  # 途中で切れても続きから再開できるように、25件ごとに保存
            print(f'   {i}/{len(jobs)} 完了（{int(time.time() - start_time)}秒）')

    show_memory('文字起こしの終了時')
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f'✅ 文字起こし完了（{int(time.time() - start_time)}秒、失敗 {errors}件）')
    print(f'🛟 空だった長いクリップを再挑戦で救えた: {len(rescued)}件 ／ それでも空: {len(still_empty)}件')
    for start_sec, how, text in rescued[:10]:
        print(f'   {format_time(start_sec)} {how}文字が出ました: 「{text[:30]}」')
    for start_sec in still_empty[:10]:
        print(f'   {format_time(start_sec)} それでも空でした')
    print(f'💾 保存しました: {CLIP_TEXT_PATH}')
else:
    print('♻️ すべて保存済みです。文字起こしはやり直しません')


### Step 5-3: 会話の形にして保存
短い声（文字起こししなかったもの）は、文字なしの発言として残します。無音で出がちな定型句は、幻覚とみなして捨てます。


In [ ]:
# 前のセルを飛ばしていないか確認（インストールのあとにランタイムが再起動すると、変数が消えます）
_missing = [name for name in ['clips', 'clip_texts', 'clip_key', 'show_dialogue', 'save_dialogue', 'PAD_SEC'] if name not in globals()]
if _missing:
    raise RuntimeError(
        f'❌ 前のセルが実行されていません（足りないもの: {_missing}）。\n'
        '   Step 1-2（設定と認証）から順に、上のセルを実行し直してください。'
    )
del _missing

import re

# 無音やノイズのとき Whisper が出しがちな定型句（短いクリップでこれだけが出たら、幻覚とみなして捨てる）
HALLUCINATION_PHRASES = ('ご視聴ありがとうございました', 'ご視聴ありがとうございます', 'チャンネル登録', 'ご清聴ありがとうございました', '字幕')
HALLUCINATION_MAX_CLIP_SEC = 5.0  # この秒数より短いクリップだけ、定型句を幻覚とみなす
MAX_CHARS_PER_SEC = 12            # 1秒にこれ以上の文字が出たら、不自然なので警告する（日本語の会話は毎秒 5〜8 文字くらい）
DROP_BLANK_TURNS = True           # 文字がない発言（短い声など）を捨てて、同じ話者をまとめ直す。False なら、文字なしの発言として残す


def clean_clip_text(text, duration):
    # クリップの文字起こしを点検する。戻り値: (使う文字, 理由)。理由は None / 'hallucination' / 'too_fast'
    plain = re.sub(r'[\s。、，．,.?？!！]', '', text)
    if duration < HALLUCINATION_MAX_CLIP_SEC and any(p in plain for p in HALLUCINATION_PHRASES):
        return '', 'hallucination'
    if duration > 0 and len(plain) / duration > MAX_CHARS_PER_SEC:
        return text, 'too_fast'
    return text, None


def merge_same_speaker(turns):
    # 間に別の話者の発言がなく、同じ話者が続くところは、1つの発言にまとめる
    merged = []
    for turn in turns:
        if merged and merged[-1]['speaker'] == turn['speaker']:
            merged[-1]['text'] += turn['text']
            merged[-1]['end'] = max(merged[-1]['end'], turn['end'])
        else:
            merged.append(dict(turn))
    return merged


turns = []
dropped, flagged = [], []
for clip in clips:
    text = ''
    if clip['transcribe']:
        raw_text = clip_texts.get(clip_key(clip), '')
        text, reason = clean_clip_text(raw_text, clip['end'] - clip['start'])
        if reason == 'hallucination':
            dropped.append((clip['start'], raw_text))
        elif reason == 'too_fast':
            flagged.append((clip['start'], len(raw_text) / (clip['end'] - clip['start'])))
    turns.append({'speaker': clip['speaker'], 'start': clip['start'], 'end': clip['end'], 'text': text})

blank_turns = sum(1 for t in turns if not t['text'].strip())
if DROP_BLANK_TURNS:
    # 話者分離が短い区間を交互に出すことがある（ちらつき）。文字がなければ内容もないので捨てる
    turns = [t for t in turns if t['text'].strip()]
turns_a2 = merge_same_speaker(turns)

print('■ Step 5: 話者で先に区切ってから文字起こし')
print(f'🕳️ 文字がない発言（短い声など）: {blank_turns}件 → ' + ('捨てて、同じ話者をまとめ直しました' if DROP_BLANK_TURNS else '残しました'))
if dropped:
    print(f'🗑️ 幻覚とみなして捨てたクリップ: {len(dropped)}件')
    for start_sec, raw_text in dropped[:10]:
        print(f'   {format_time(start_sec)} 「{raw_text}」')
if flagged:
    print(f'⚠️ 文字数が多すぎる（毎秒 {MAX_CHARS_PER_SEC} 文字超）クリップ: {len(flagged)}件')
    for start_sec, rate in flagged[:10]:
        print(f'   {format_time(start_sec)} 毎秒 {rate:.1f} 文字')

show_dialogue(turns_a2)
save_dialogue(turns_a2, f'result_A2_pad{PAD_SEC}.txt')
